###Sử dụng google colab nếu máy tính cá nhân bạn không đủ 16GB RAM GPU
###Use google colab if your laptop or pc don't have of 16GB RAM GPU


In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
!pip install transformers peft bitsandbytes accelerate streamlit google-generativeai pylatexenc pyngrok
!pip install -q streamlit pyngrok google-generativeai transformers accelerate bitsandbytes peft pillow pylatexenc

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 162.6/162.6 kB 12.9 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 13.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.2/9.2 MB 66.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.3/11.3 MB 64.1 MB/s eta 0:00:00
  Created wheel for pylatexenc: filename=pylatexenc-2.10-py3-none-any.whl size=136817 sha256=c0430f9d0b86252889233c467dcd928819a7f34ad27685a87bd6da5687d6d6b4
  Stored in directory: /root/.cache/pip/wheels/06/3e/78/fa1588c1ae991bbfd814af2bcac6cef7a178beee1939180d46
Successfully built pylatexenc


In [3]:
%%bash
mkdir -p /content/.streamlit
cat > /content/.streamlit/secrets.toml << 'EOF'
GEMINI_API_KEY = "API_KEY"
EOF

In [4]:
%%writefile app.py
import re, torch, streamlit as st
from PIL import Image
import google.generativeai as genai
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel
from pylatexenc.latex2text import LatexNodes2Text
from transformers import StoppingCriteria, StoppingCriteriaList

#model và adapter
MODEL_NAME = "Qwen/Qwen2.5-Math-7B"
ADAPTER_DIR = "/content/drive/MyDrive/Khoa_luan_2025_2026/qwen2_5_math_adapter"

#Gemini OCR
gemini = genai.GenerativeModel("gemini-3-flash-preview")
genai.configure(api_key=st.secrets["GEMINI_API_KEY"])

def extract_latex(data, is_image):
    prompt = """
    Bạn là một hệ thống OCR chuyên dụng giúp chuyển đổi hình ảnh hoặc văn bản của câu hỏi chứa đề bài toán học thành dạng raw LaTeX.

    Yêu cầu BẮT BUỘC:
    1. Tất cả các số,  công thức toán học trong văn bản hoặc hình ảnh có bảng đều phải được chuyển đổi chính xác và đặt trong cặp dấu $$.
    2. Chỉ trả về đúng chuỗi raw LaTeX, không kèm theo bất kỳ văn bản giải thích nào khác ngoài nội dung có trong ảnh hoặc văn bản.
    3. Hãy loại bỏ tất cả những văn bản thừa, không liên quan đến bài toán (nếu có).
    """
    try:
        if is_image:
            img = Image.open(data).convert("RGB")
            resp = gemini.generate_content(
                [prompt, img],
                generation_config=genai.types.GenerationConfig(temperature=0.0)
            )
        else:
            input = str(data)
            resp = gemini.generate_content(
                [prompt, input],
                generation_config=genai.types.GenerationConfig(temperature=0.0)
            )
        return resp.text.strip()
    except Exception as e:
        return f"Lỗi OCR: {e}"

@st.cache_resource
def load_qwen():

    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_use_double_quant=True,
        bnb_4bit_compute_dtype=torch.bfloat16
    )

    tokenizer = AutoTokenizer.from_pretrained(
        ADAPTER_DIR,
        trust_remote_code=True
    )

    if tokenizer.pad_token_id is None:
        tokenizer.pad_token_id = tokenizer.eos_token_id

    # load base model 4-bit QLoRA inference
    base_model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME,
        quantization_config=bnb_config,
        torch_dtype=torch.bfloat16,
        device_map="auto",
        trust_remote_code=True
    )

    # gắn adapter đã fine-tune
    model = PeftModel.from_pretrained(
        base_model,
        ADAPTER_DIR,
        is_trainable=False)

    model.eval()
    return tokenizer, model

def build_prompt(latex):
    return f"Hãy giúp tôi giải bài sau:\n{latex}\n"

def clean_after_dapan(text):
    text = str(text).strip()

    text = re.sub(r"\s+", " ", text).strip()

    first = re.search(r"/dapan\w*", text)
    if not first:
        return text

    before = text[:first.start()].strip()
    after = text[first.end():].strip()

    next_match = re.search(r"/dapan\w*", after)
    if next_match:
        after = after[:next_match.start()].strip()

    next_text_label = re.search(r"\bĐáp án\s*:", after, flags=re.IGNORECASE)
    if next_text_label:
        after = after[:next_text_label.start()].strip()

    cleaned = before + " /dapan " + after
    cleaned = re.sub(r"\s+", " ", cleaned).strip()

    return cleaned



def generate_answer(latex, seed=42):
    tokenizer, model = load_qwen()

    class StopOnString(StoppingCriteria):
        def __init__(self, tokenizer, stop_string):
            self.stop_ids = tokenizer(stop_string, return_tensors="pt").input_ids[0]

        def __call__(self, input_ids, scores, **kwargs):
            if len(input_ids[0]) < len(self.stop_ids):
                return False
            return (input_ids[0][-len(self.stop_ids):] == self.stop_ids.to(input_ids.device)).all()

    stopping_criteria = StoppingCriteriaList([
        StopOnString(tokenizer, "/dapan")
    ])
    torch.manual_seed(seed)
    prompt = build_prompt(latex)
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    outputs = model.generate(
        **inputs,top_p=0.9,
        max_new_tokens=512,
        temperature=0.0,
        stopping_criteria=stopping_criteria,
        do_sample=False, # do_sample=False
        eos_token_id=tokenizer.eos_token_id,
        pad_token_id=tokenizer.pad_token_id,
        early_stopping=True)

    output = tokenizer.decode(outputs[0], skip_special_tokens=True)
    output = output[len(prompt):].strip()

    output = clean_after_dapan(output)

    output = re.sub(r'\s*/buoc1\s*', '\n\n**Bước 1:** ', output)
    output = re.sub(r'\s*/buoc2\s*', '\n\n**Bước 2:** ', output)
    output = re.sub(r'\s*/buoc3\s*', '\n\n**Bước 3:** ', output)
    output = re.sub(r'\s*/buoc4\s*', '\n\n**Bước 4:** ', output)
    output = re.sub(r'\s*/buoc5\s*', '\n\n**Bước 5:** ', output)
    output = re.sub(r'\s*/buoc6\s*', '\n\n**Bước 6:** ', output)
    output = re.sub(r'\s*/buoc7\s*', '\n\n**Bước 7:** ', output)
    output = re.sub(r'\s*/buoc8\s*', '\n\n**Bước 8:** ', output)
    output = re.sub(r'\s*/buoc9\s*', '\n\n**Bước 9:** ', output)
    output = re.sub(r'\s*/buoc10\s*', '\n\n**Bước 10:** ', output)
    output = re.sub(r'\s*/buoc11\s*', '\n\n**Bước 11:** ', output)
    output = re.sub(r'\s*/ketluan\s*', '\n\n**Kết luận:** ', output)
    output = re.sub(r'\s*/dapan\s*', '\n\n**Đáp án:** ', output)
    return output



def latex_to_text(latex):
    try:
        return LatexNodes2Text().latex_to_text(latex)
    except Exception:
        return latex

# Giao diện Streamlit
st.title("Demo")
mode = st.radio("Chọn đầu vào:", ["Ảnh", "Văn bản"])

if mode == "Ảnh":
    file = st.file_uploader("Tải ảnh đề toán", type=["png","jpg","jpeg"])
    if file:
        st.image(file)
        if st.button("Giải bài"):
            with st.spinner("Chuyển đổi ảnh sang LaTeX..."):
                latex_in = extract_latex(file, True)
            st.markdown(f"**LaTeX đề bài:** `{latex_in}`")
            with st.spinner("Đang giải..."):
                ans_latex = generate_answer(latex_in)
            st.markdown("**Đáp án:** " + latex_to_text(ans_latex))


elif mode == "Văn bản":
    text = st.text_area("Nhập đề bài", height=180)
    if st.button("Giải bài"):
        if text.strip() == "":
            st.warning("Vui lòng nhập đề bài.")
        else:
            with st.spinner("Chuyển văn bản sang LaTeX..."):
                latex_in = extract_latex(text, False)
            st.markdown(f"**LaTeX đề bài:** `{latex_in}`")
            with st.spinner("Đang giải..."):
                ans_latex = generate_answer(latex_in)
            st.markdown("**Đáp án:** " + latex_to_text(ans_latex))



Writing app.py


In [5]:
!pkill -f streamlit
!streamlit run app.py --server.port 8501 --server.address 0.0.0.0 --server.headless true > streamlit.log 2>&1 &


In [7]:
from pyngrok import ngrok
ngrok.set_auth_token("3DfN0KFr8KFP2fWToHeOnyToZoz_27N5ptZTZLN2FLccmbnYr")
# ngrok.kill()
public_url = ngrok.connect(8501)
print("Link demo:", public_url)


Link demo: NgrokTunnel: "https://epidural-handed-rehab.ngrok-free.dev" -> "http://localhost:8501"
